# See What Matters: Skin Lesion Robustness (Fast Run)

Same experiment as the full notebook but trimmed for speed:
- 1000 train images (stratified sample) instead of full dataset
- 3 epochs instead of 8
- ~5 min total on Colab GPU

Fine-tunes MobileNetV2 two ways (baseline vs domain-augmented), then tests robustness across 11 perturbations.

In [ ]:
!pip install -q torch torchvision datasets pillow matplotlib numpy scikit-learn

In [ ]:
import os, io, time, random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load data (subsample for speed)

In [ ]:
from datasets import load_dataset

print('Loading dataset...')
ds = load_dataset('ahmed-ai/skin-lesions-classification-dataset')

train_labels = [row['label'] for row in ds['train']]
label_names = sorted(set(train_labels))
NUM_CLASSES = len(label_names)
label2idx = {l: i for i, l in enumerate(label_names)}
idx2label = {i: l for l, i in label2idx.items()}

print(f'Full train: {len(ds["train"])}, test: {len(ds["test"])}')
print(f'{NUM_CLASSES} classes: {label_names}')

# stratified subsample: 1000 train images, keep test as-is
TRAIN_SUBSET = 1000
indices_by_class = {}
for i, lbl in enumerate(train_labels):
    indices_by_class.setdefault(lbl, []).append(i)

per_class = max(TRAIN_SUBSET // NUM_CLASSES, 10)
subset_indices = []
for cls, idxs in indices_by_class.items():
    random.shuffle(idxs)
    subset_indices.extend(idxs[:per_class])
random.shuffle(subset_indices)

train_sub = ds['train'].select(subset_indices)
print(f'Using {len(train_sub)} train images (stratified sample)')

## 2. Perturbations

In [ ]:
def shift_skin_tone(img, factor):
    arr = np.asarray(img.convert('RGB'), dtype=np.float32)
    lin = np.power(np.clip(arr / 255.0, 0, 1), 2.2)
    out = lin * factor
    rgb = (np.clip(np.power(out, 1/2.2), 0, 1) * 255).astype(np.uint8)
    return Image.fromarray(rgb)

def warm_lighting(img, s=0.15):
    a = np.asarray(img.convert('RGB'), dtype=np.float32)
    a[...,0] = np.clip(a[...,0]*(1+s), 0, 255)
    a[...,2] = np.clip(a[...,2]*(1-s), 0, 255)
    return Image.fromarray(a.astype(np.uint8))

def cool_lighting(img, s=0.15):
    a = np.asarray(img.convert('RGB'), dtype=np.float32)
    a[...,0] = np.clip(a[...,0]*(1-s), 0, 255)
    a[...,2] = np.clip(a[...,2]*(1+s), 0, 255)
    return Image.fromarray(a.astype(np.uint8))

def gauss_noise(img, sigma=15.0):
    a = np.asarray(img.convert('RGB'), dtype=np.float32)
    return Image.fromarray(np.clip(a + np.random.normal(0, sigma, a.shape), 0, 255).astype(np.uint8))

def motion_blur(img, k=9):
    a = np.asarray(img.convert('RGB'), dtype=np.float32)
    k = max(3, k|1); p = k//2
    padded = np.pad(a, ((0,0),(p,p),(0,0)), mode='edge')
    out = sum(padded[:, i:i+a.shape[1], :] for i in range(k)) / k
    return Image.fromarray(out.astype(np.uint8))

def jpeg_compress(img, q=25):
    buf = io.BytesIO()
    img.convert('RGB').save(buf, format='JPEG', quality=q)
    buf.seek(0)
    return Image.open(buf).copy()

PERTURBATIONS = {
    'darker_skin':    lambda im: shift_skin_tone(im, 0.55),
    'lighter_skin':   lambda im: shift_skin_tone(im, 1.45),
    'low_light':      lambda im: ImageEnhance.Brightness(im).enhance(0.55),
    'harsh_light':    lambda im: ImageEnhance.Brightness(ImageEnhance.Contrast(im).enhance(1.4)).enhance(1.25),
    'warm_white_bal': lambda im: warm_lighting(im, 0.20),
    'cool_white_bal': lambda im: cool_lighting(im, 0.20),
    'motion_blur':    lambda im: motion_blur(im, 11),
    'out_of_focus':   lambda im: im.filter(ImageFilter.GaussianBlur(radius=2.5)),
    'sensor_noise':   lambda im: gauss_noise(im, 18.0),
    'jpeg_artifacts': lambda im: jpeg_compress(im, 20),
    'off_axis':       lambda im: im.rotate(15, resample=Image.BILINEAR, fillcolor=(0,0,0)),
}
print(f'{len(PERTURBATIONS)} perturbations defined')

## 3. Transforms + Datasets

In [ ]:
IMG_SIZE = 224
normalize = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

baseline_tfm = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), normalize,
])

class DomainAugTransform:
    def __init__(self):
        self.post = transforms.Compose([
            transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(), normalize,
        ])
    def __call__(self, img):
        if random.random() < 0.5:
            fn = random.choice(list(PERTURBATIONS.values()))
            try: img = fn(img)
            except: pass
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.3:
            img = img.rotate(random.uniform(-20,20), resample=Image.BILINEAR, fillcolor=(0,0,0))
        return self.post(img)

test_tfm = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), normalize,
])

class SkinDS(Dataset):
    def __init__(self, hf_ds, tfm, l2i):
        self.data, self.tfm, self.l2i = hf_ds, tfm, l2i
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        r = self.data[i]
        return self.tfm(r['image'].convert('RGB')), self.l2i[r['label']]

class PerturbedDS(Dataset):
    def __init__(self, hf_ds, pert_fn, tfm, l2i):
        self.data, self.pert, self.tfm, self.l2i = hf_ds, pert_fn, tfm, l2i
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        r = self.data[i]
        img = self.pert(r['image'].convert('RGB'))
        return self.tfm(img), self.l2i[r['label']]

BS = 32
train_bl_loader = DataLoader(SkinDS(train_sub, baseline_tfm, label2idx), batch_size=BS, shuffle=True, num_workers=2)
train_aug_loader = DataLoader(SkinDS(train_sub, DomainAugTransform(), label2idx), batch_size=BS, shuffle=True, num_workers=2)
test_loader = DataLoader(SkinDS(ds['test'], test_tfm, label2idx), batch_size=BS, shuffle=False, num_workers=2)
print(f'Train batches: {len(train_bl_loader)}, Test batches: {len(test_loader)}')

## 4. Model + Training

In [ ]:
def make_model(nc):
    net = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    for p in net.features.parameters(): p.requires_grad = False
    net.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(net.last_channel, nc))
    return net.to(device)

def train(model, loader, epochs=3, lr=1e-3):
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    hist = {'loss':[], 'acc':[]}
    for ep in range(epochs):
        model.train()
        rloss = correct = total = 0
        t0 = time.time()
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            out = model(imgs)
            loss = crit(out, labels)
            loss.backward(); opt.step()
            rloss += loss.item()*imgs.size(0)
            correct += (out.argmax(1)==labels).sum().item()
            total += labels.size(0)
        hist['loss'].append(rloss/total)
        hist['acc'].append(correct/total)
        print(f'  Epoch {ep+1}/{epochs} -- loss:{rloss/total:.4f} acc:{correct/total:.4f} ({time.time()-t0:.1f}s)')
    return hist

EPOCHS = 3

print('--- Training BASELINE ---')
bl_model = make_model(NUM_CLASSES)
bl_hist = train(bl_model, train_bl_loader, EPOCHS)

print('\n--- Training AUGMENTED ---')
aug_model = make_model(NUM_CLASSES)
aug_hist = train(aug_model, train_aug_loader, EPOCHS)

## 5. Training curves

In [ ]:
fig, (a1,a2) = plt.subplots(1,2,figsize=(12,4))
a1.plot(bl_hist['loss'], label='Baseline'); a1.plot(aug_hist['loss'], label='Augmented')
a1.set_xlabel('Epoch'); a1.set_ylabel('Loss'); a1.set_title('Training Loss'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(bl_hist['acc'], label='Baseline'); a2.plot(aug_hist['acc'], label='Augmented')
a2.set_xlabel('Epoch'); a2.set_ylabel('Accuracy'); a2.set_title('Training Accuracy'); a2.legend(); a2.grid(alpha=0.3)
fig.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

## 6. Evaluate on clean + perturbed test sets

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for imgs, lbl in loader:
        out = model(imgs.to(device))
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(lbl.numpy())
    return np.array(labels), np.array(preds)

print('=== Clean test set ===')
bl_lbl, bl_pred = evaluate(bl_model, test_loader)
aug_lbl, aug_pred = evaluate(aug_model, test_loader)
bl_acc = accuracy_score(bl_lbl, bl_pred)
aug_acc = accuracy_score(aug_lbl, aug_pred)
print(f'Baseline accuracy:  {bl_acc:.4f}')
print(f'Augmented accuracy: {aug_acc:.4f}')
print()
print('Baseline report:')
print(classification_report(bl_lbl, bl_pred, target_names=label_names))
print('Augmented report:')
print(classification_report(aug_lbl, aug_pred, target_names=label_names))

In [ ]:
results = {}
for pname, pfn in PERTURBATIONS.items():
    print(f'Testing: {pname}...', end=' ')
    ploader = DataLoader(PerturbedDS(ds['test'], pfn, test_tfm, label2idx), batch_size=BS, shuffle=False, num_workers=2)
    _, bl_pp = evaluate(bl_model, ploader)
    _, aug_pp = evaluate(aug_model, ploader)
    results[pname] = {
        'bl_acc': accuracy_score(bl_lbl, bl_pp),
        'aug_acc': accuracy_score(aug_lbl, aug_pp),
        'bl_agree': np.mean(bl_pp == bl_pred),
        'aug_agree': np.mean(aug_pp == aug_pred),
    }
    print(f'BL acc:{results[pname]["bl_acc"]:.4f}  AUG acc:{results[pname]["aug_acc"]:.4f}  '
          f'BL agree:{results[pname]["bl_agree"]:.4f}  AUG agree:{results[pname]["aug_agree"]:.4f}')
print('Done!')

## 7. Robustness charts

In [ ]:
names = list(results.keys())
bl_a = [results[n]['bl_acc'] for n in names]
aug_a = [results[n]['aug_acc'] for n in names]
bl_ag = [results[n]['bl_agree'] for n in names]
aug_ag = [results[n]['aug_agree'] for n in names]
x = np.arange(len(names)); w = 0.38

fig, (a1,a2) = plt.subplots(2,1,figsize=(12,9))

a1.bar(x-w/2, bl_a, w, label='Baseline', color='#e74c3c')
a1.bar(x+w/2, aug_a, w, label='Augmented', color='#2ecc71')
a1.axhline(bl_acc, color='#e74c3c', ls=':', alpha=0.5, label=f'BL clean ({bl_acc:.3f})')
a1.axhline(aug_acc, color='#2ecc71', ls=':', alpha=0.5, label=f'AUG clean ({aug_acc:.3f})')
a1.set_ylabel('Accuracy'); a1.set_title('Accuracy under perturbation')
a1.set_xticks(x); a1.set_xticklabels(names, rotation=35, ha='right')
a1.legend(loc='lower right', fontsize=9); a1.grid(axis='y', alpha=0.3); a1.set_ylim(0,1.05)

a2.bar(x-w/2, bl_ag, w, label='Baseline', color='#e74c3c')
a2.bar(x+w/2, aug_ag, w, label='Augmented', color='#2ecc71')
a2.axhline(1.0, color='gray', ls=':', lw=0.8)
a2.set_ylabel('Agreement with clean pred'); a2.set_title('Prediction stability (higher = more robust)')
a2.set_xticks(x); a2.set_xticklabels(names, rotation=35, ha='right')
a2.legend(loc='lower right', fontsize=9); a2.grid(axis='y', alpha=0.3); a2.set_ylim(0,1.05)

fig.tight_layout(); plt.savefig('robustness_comparison.png', dpi=150); plt.show()

## 8. Summary + Perturbation gallery

In [ ]:
print(f'{"Perturbation":<18} {"BL Acc":>8} {"AUG Acc":>8} {"Diff":>8} {"BL Agree":>10} {"AUG Agree":>10} {"Diff":>8}')
print('-'*80)
for n in names:
    r = results[n]
    print(f'{n:<18} {r["bl_acc"]:>8.4f} {r["aug_acc"]:>8.4f} {r["aug_acc"]-r["bl_acc"]:>+8.4f} '
          f'{r["bl_agree"]:>10.4f} {r["aug_agree"]:>10.4f} {r["aug_agree"]-r["bl_agree"]:>+8.4f}')
print('-'*80)
print(f'{"AVERAGE":<18} {np.mean(bl_a):>8.4f} {np.mean(aug_a):>8.4f} {np.mean(aug_a)-np.mean(bl_a):>+8.4f} '
      f'{np.mean(bl_ag):>10.4f} {np.mean(aug_ag):>10.4f} {np.mean(aug_ag)-np.mean(bl_ag):>+8.4f}')
print(f'\nClean test -- Baseline: {bl_acc:.4f}, Augmented: {aug_acc:.4f}')

In [ ]:
sample = ds['test'][0]['image'].convert('RGB')
fig, axes = plt.subplots(3, 4, figsize=(16,10))
axes = axes.flatten()
axes[0].imshow(sample); axes[0].set_title('Original', fontweight='bold'); axes[0].axis('off')
for i, (nm, fn) in enumerate(PERTURBATIONS.items()):
    axes[i+1].imshow(fn(sample)); axes[i+1].set_title(nm, fontsize=10); axes[i+1].axis('off')
fig.suptitle('All 11 perturbations on one image', fontsize=14)
fig.tight_layout(); plt.savefig('perturbation_gallery.png', dpi=150); plt.show()

In [ ]:
fig, (a1,a2) = plt.subplots(1,2,figsize=(14,5))
cm1 = confusion_matrix(bl_lbl, bl_pred)
cm2 = confusion_matrix(aug_lbl, aug_pred)
a1.imshow(cm1, cmap='Blues'); a1.set_title(f'Baseline (acc={bl_acc:.3f})')
a1.set_xlabel('Predicted'); a1.set_ylabel('Actual')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        a1.text(j,i,str(cm1[i][j]),ha='center',va='center',fontsize=8)
a2.imshow(cm2, cmap='Greens'); a2.set_title(f'Augmented (acc={aug_acc:.3f})')
a2.set_xlabel('Predicted'); a2.set_ylabel('Actual')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        a2.text(j,i,str(cm2[i][j]),ha='center',va='center',fontsize=8)
fig.tight_layout(); plt.savefig('confusion_matrices.png', dpi=150); plt.show()
print('All done! Download PNGs from the sidebar.')